# Pairwise Bootstrap Comparison — PsychioNet Pretraining

Compares **BYOL**, **SimSiam**, and **DualBranch-no-scale** models pretrained on PsychioNet,
evaluated on WESAD via 15-fold LOSO fine-tuning.

**Run `generate_predictions_psychionet.py` first** to produce the prediction pkl files.

Bootstrap procedure (n=1000):
- Resample N test-sample indices with replacement per iteration
- Compute macro-F1 difference between each model pair
- Two-tailed p-value = fraction of bootstrap iterations where |Δ*| ≥ |Δ_observed|
- 95% CI = [2.5th, 97.5th] percentile of Δ* distribution

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import f1_score, accuracy_score
from itertools import combinations

REPO     = "/home/s223149341/SSL-invariance-Subject_Project_model/subject_aware_contrastive_learning"
PRED_DIR = os.path.join(REPO, "save/predictions")

MODEL_NAMES = {
    "byol":                 "BYOL",
    "simsiam":              "SimSiam",
    "dual_branch_no_scale": "DualBranch-no-scale",
}

N_BOOTSTRAP  = 1000
RANDOM_SEED  = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Imports OK")

## 1. Load Predictions

In [ ]:
preds = {}
for key, label in MODEL_NAMES.items():
    path = os.path.join(PRED_DIR, f"psychionet_{key}_preds.pkl")
    if not os.path.isfile(path):
        print(f"WARNING: {path} not found — run generate_predictions_psychionet.py first")
        continue
    with open(path, "rb") as f:
        data = pickle.load(f)
    preds[key] = data
    y_true, y_pred = data["y_true"], data["y_pred"]
    f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    n_folds = len(np.unique(data["fold_id"]))
    print(f"  {label:<25}  F1={f1:.4f}  Acc={acc:.4f}  n={len(y_true)}  folds={n_folds}")

assert len(preds) >= 2, "Need at least 2 models to compare."

# Sanity: all models must share the same y_true
keys = list(preds.keys())
ref_true = preds[keys[0]]["y_true"]
for k in keys[1:]:
    assert np.array_equal(preds[k]["y_true"], ref_true), \
        f"y_true mismatch between {keys[0]} and {k}!"
print("\ny_true sanity check passed.")

## 2. Per-Fold F1 Scores

In [ ]:
fold_rows = []
for key, label in MODEL_NAMES.items():
    if key not in preds:
        continue
    data = preds[key]
    for fold_id in np.unique(data["fold_id"]):
        mask   = data["fold_id"] == fold_id
        y_t    = data["y_true"][mask]
        y_p    = data["y_pred"][mask]
        fold_f1  = f1_score(y_t, y_p, average="macro", zero_division=0)
        fold_acc = accuracy_score(y_t, y_p)
        fold_rows.append({"Model": label, "Fold": fold_id, "F1": fold_f1, "Acc": fold_acc})

fold_df = pd.DataFrame(fold_rows)

summary = fold_df.groupby("Model")[["F1", "Acc"]].agg(["mean", "std"]).round(4)
summary.columns = ["F1_mean", "F1_std", "Acc_mean", "Acc_std"]
print("Per-fold summary:")
display(summary)

# Box plot of per-fold F1
fig, ax = plt.subplots(figsize=(8, 4))
model_order = [MODEL_NAMES[k] for k in keys]
data_by_model = [fold_df[fold_df["Model"] == m]["F1"].values for m in model_order]
bp = ax.boxplot(data_by_model, labels=model_order, patch_artist=True, notch=False)
colors = ["#4C72B0", "#DD8452", "#55A868"]
for patch, color in zip(bp["boxes"], colors[:len(bp["boxes"])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("Macro F1")
ax.set_title("Per-Fold Macro F1 by Model (PsychioNet pretraining, WESAD LOSO)")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PRED_DIR, "per_fold_f1_boxplot.png"), dpi=150)
plt.show()

## 3. Pairwise Bootstrap (n=1000)

For each pair (A, B):
- Observed Δ = F1(A) − F1(B)
- 1000 bootstrap iterations: resample N samples with replacement, compute Δ*
- Two-tailed p-value = fraction of |Δ*| ≥ |Δ|
- 95% CI = [2.5th, 97.5th] percentile of Δ*

In [ ]:
def bootstrap_f1_diff(y_true, y_pred_a, y_pred_b,
                      n_iter=1000, rng=None, metric="f1"):
    """
    Two-tailed paired bootstrap test of F1 difference between model A and B.

    Returns
    -------
    dict with:
        observed_diff  : Δ = metric(A) − metric(B) on full sample
        boot_diffs     : (n_iter,) array of Δ* from each bootstrap resample
        p_value        : two-tailed p-value
        ci_low, ci_high: 95% CI of Δ*
        f1_a, f1_b     : full-sample metrics
    """
    if rng is None:
        rng = np.random.default_rng(42)

    def compute(yt, yp):
        if metric == "f1":
            return f1_score(yt, yp, average="macro", zero_division=0)
        return accuracy_score(yt, yp)

    m_a = compute(y_true, y_pred_a)
    m_b = compute(y_true, y_pred_b)
    obs_diff = m_a - m_b

    N = len(y_true)
    boot_diffs = np.empty(n_iter)
    for i in range(n_iter):
        idx = rng.integers(0, N, size=N)
        boot_diffs[i] = compute(y_true[idx], y_pred_a[idx]) - compute(y_true[idx], y_pred_b[idx])

    # two-tailed: proportion of |Δ*| ≥ |Δ|
    p_val  = np.mean(np.abs(boot_diffs) >= np.abs(obs_diff))
    ci_low = np.percentile(boot_diffs, 2.5)
    ci_high= np.percentile(boot_diffs, 97.5)

    return {
        "observed_diff": obs_diff,
        "boot_diffs":    boot_diffs,
        "p_value":       p_val,
        "ci_low":        ci_low,
        "ci_high":       ci_high,
        "metric_a":      m_a,
        "metric_b":      m_b,
    }

In [ ]:
print(f"Running pairwise bootstrap (n={N_BOOTSTRAP}) ...\n")

bootstrap_results = {}
y_true = ref_true   # shared ground truth

for key_a, key_b in combinations(keys, 2):
    if key_a not in preds or key_b not in preds:
        continue
    label_a = MODEL_NAMES[key_a]
    label_b = MODEL_NAMES[key_b]
    pair    = f"{label_a} vs {label_b}"

    print(f"  {pair} ...", end=" ")
    res = bootstrap_f1_diff(
        y_true,
        preds[key_a]["y_pred"],
        preds[key_b]["y_pred"],
        n_iter=N_BOOTSTRAP,
        rng=rng,
        metric="f1",
    )
    bootstrap_results[pair] = res
    print(f"Δ={res['observed_diff']:+.4f}  p={res['p_value']:.4f}  "
          f"95%CI=[{res['ci_low']:+.4f}, {res['ci_high']:+.4f}]")

print("\nDone.")

## 4. Results Table

In [ ]:
rows = []
for pair, res in bootstrap_results.items():
    a_name, b_name = pair.split(" vs ")
    sig = ""
    if   res["p_value"] < 0.001: sig = "***"
    elif res["p_value"] < 0.01:  sig = "**"
    elif res["p_value"] < 0.05:  sig = "*"
    rows.append({
        "Model A":       a_name,
        "F1 (A)":        round(res["metric_a"], 4),
        "Model B":       b_name,
        "F1 (B)":        round(res["metric_b"], 4),
        "Δ (A−B)":       round(res["observed_diff"], 4),
        "95% CI low":    round(res["ci_low"], 4),
        "95% CI high":   round(res["ci_high"], 4),
        "p-value":       round(res["p_value"], 4),
        "Sig.": sig,
    })

results_df = pd.DataFrame(rows)
print("Pairwise Bootstrap Results (macro F1, n=1000, two-tailed)")
print("* p<0.05  ** p<0.01  *** p<0.001")
display(results_df)

results_df.to_csv(os.path.join(PRED_DIR, "bootstrap_results.csv"), index=False)
print(f"Saved → {os.path.join(PRED_DIR, 'bootstrap_results.csv')}")

## 5. Bootstrap Distribution Plots

In [ ]:
n_pairs = len(bootstrap_results)
fig, axes = plt.subplots(1, n_pairs, figsize=(6 * n_pairs, 4), sharey=False)
if n_pairs == 1:
    axes = [axes]

for ax, (pair, res) in zip(axes, bootstrap_results.items()):
    diffs = res["boot_diffs"]
    obs   = res["observed_diff"]
    p     = res["p_value"]

    ax.hist(diffs, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
    ax.axvline(obs,  color="crimson",  lw=2, label=f"Observed Δ={obs:+.4f}")
    ax.axvline(-obs, color="crimson",  lw=2, linestyle="--")
    ax.axvline(res["ci_low"],  color="orange", lw=1.5, linestyle=":", label="95% CI")
    ax.axvline(res["ci_high"], color="orange", lw=1.5, linestyle=":")
    ax.axvline(0, color="black", lw=1, linestyle="-", alpha=0.4)

    ax.set_title(pair, fontsize=11, fontweight="bold")
    ax.set_xlabel("Δ F1 (A − B)")
    ax.set_ylabel("Frequency")

    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    ax.text(0.98, 0.97, f"p={p:.4f} {sig}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=10, color="crimson",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    ax.legend(fontsize=8)

plt.suptitle(f"Bootstrap Distributions (n={N_BOOTSTRAP}) — PsychioNet pretraining → WESAD LOSO",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PRED_DIR, "bootstrap_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {os.path.join(PRED_DIR, 'bootstrap_distributions.png')}")

## 6. Per-Fold Bootstrap (within each LOSO fold)

In [ ]:
fold_bootstrap_rows = []
fold_ids = np.unique(ref_true if False else preds[keys[0]]["fold_id"])

for key_a, key_b in combinations(keys, 2):
    if key_a not in preds or key_b not in preds:
        continue
    label_a = MODEL_NAMES[key_a]
    label_b = MODEL_NAMES[key_b]

    for fold_id in fold_ids:
        mask_a = preds[key_a]["fold_id"] == fold_id
        mask_b = preds[key_b]["fold_id"] == fold_id

        yt  = preds[key_a]["y_true"][mask_a]
        ypa = preds[key_a]["y_pred"][mask_a]
        ypb = preds[key_b]["y_pred"][mask_b]

        if len(yt) == 0:
            continue

        res = bootstrap_f1_diff(yt, ypa, ypb,
                                n_iter=N_BOOTSTRAP, rng=rng, metric="f1")
        fold_bootstrap_rows.append({
            "Pair":       f"{label_a} vs {label_b}",
            "Fold":       fold_id,
            "F1 (A)":     round(res["metric_a"], 4),
            "F1 (B)":     round(res["metric_b"], 4),
            "Δ (A−B)":   round(res["observed_diff"], 4),
            "p-value":    round(res["p_value"], 4),
            "CI low":     round(res["ci_low"], 4),
            "CI high":    round(res["ci_high"], 4),
        })

fold_boot_df = pd.DataFrame(fold_bootstrap_rows)
display(fold_boot_df)

fold_boot_df.to_csv(os.path.join(PRED_DIR, "bootstrap_per_fold.csv"), index=False)
print(f"Saved → {os.path.join(PRED_DIR, 'bootstrap_per_fold.csv')}")

## 7. Significance Summary Heatmap

In [ ]:
all_labels = [MODEL_NAMES[k] for k in keys]
n = len(all_labels)
pval_matrix = np.full((n, n), np.nan)
diff_matrix = np.full((n, n), np.nan)

for i, key_a in enumerate(keys):
    for j, key_b in enumerate(keys):
        if i == j:
            continue
        label_a = MODEL_NAMES[key_a]
        label_b = MODEL_NAMES[key_b]
        pair_fwd = f"{label_a} vs {label_b}"
        pair_bwd = f"{label_b} vs {label_a}"
        if pair_fwd in bootstrap_results:
            res = bootstrap_results[pair_fwd]
            pval_matrix[i, j] = res["p_value"]
            diff_matrix[i, j] = res["observed_diff"]
        elif pair_bwd in bootstrap_results:
            res = bootstrap_results[pair_bwd]
            pval_matrix[i, j] = res["p_value"]
            diff_matrix[i, j] = -res["observed_diff"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# p-value matrix
im0 = axes[0].imshow(pval_matrix, vmin=0, vmax=0.1, cmap="RdYlGn_r")
axes[0].set_xticks(range(n)); axes[0].set_xticklabels(all_labels, rotation=30, ha="right")
axes[0].set_yticks(range(n)); axes[0].set_yticklabels(all_labels)
for i in range(n):
    for j in range(n):
        if not np.isnan(pval_matrix[i, j]):
            axes[0].text(j, i, f"{pval_matrix[i,j]:.3f}",
                         ha="center", va="center", fontsize=10)
axes[0].set_title("p-value (row vs col)", fontweight="bold")
plt.colorbar(im0, ax=axes[0])

# Δ F1 matrix
vmax = max(0.01, np.nanmax(np.abs(diff_matrix)))
im1 = axes[1].imshow(diff_matrix, vmin=-vmax, vmax=vmax, cmap="coolwarm")
axes[1].set_xticks(range(n)); axes[1].set_xticklabels(all_labels, rotation=30, ha="right")
axes[1].set_yticks(range(n)); axes[1].set_yticklabels(all_labels)
for i in range(n):
    for j in range(n):
        if not np.isnan(diff_matrix[i, j]):
            axes[1].text(j, i, f"{diff_matrix[i,j]:+.3f}",
                         ha="center", va="center", fontsize=10)
axes[1].set_title("Δ F1 (row − col)", fontweight="bold")
plt.colorbar(im1, ax=axes[1])

plt.suptitle("Pairwise Bootstrap — PsychioNet pretraining → WESAD LOSO",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PRED_DIR, "bootstrap_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {os.path.join(PRED_DIR, 'bootstrap_heatmap.png')}")